# Chapter 1 — A Provider-Neutral Model Boundary

This chapter constructs the first cumulative Checkpoint: typed messages, versioned content blocks, a deterministic adapter, and one explicitly configured OpenAI-compatible adapter.

## Goal and Previous Limitation

There is no preceding Checkpoint. The historical prototypes mix provider dictionaries, ambient credentials, display calls, and agent state. Our first slice establishes a narrow model seam that later chapters can evolve without importing those concerns.

## Conceptual Model

`AgentMessage` is the harness-facing input. `to_model_messages` validates and narrows it to `ModelMessage`. A `ModelAdapter` emits provider-neutral deltas, while `complete` assembles exactly one typed `ModelResult`. Provider request dictionaries exist only inside the production adapter.

```text
AgentMessage -> ModelMessage -> ModelAdapter -> ModelEvent -> ModelResult
                              ^
             scripted adapter | OpenAI-compatible adapter
```

## Minimal Execution

The following Export Cell is the complete model module. Assigning the source keeps the notebook executable; the tagged value, rather than notebook outputs, becomes the Checkpoint file.

In [ ]:
MODEL_SOURCE = r'''"""Provider-neutral messages, model events, and model adapters."""

from __future__ import annotations

from collections.abc import AsyncIterator, Mapping, Sequence
from dataclasses import dataclass, field
from enum import Enum
import json
from types import MappingProxyType
from typing import Literal, Protocol, TypeAlias, cast
from urllib.parse import urlparse

import openai


class Role(str, Enum):
    SYSTEM = 'system'
    USER = 'user'
    ASSISTANT = 'assistant'


class StopReason(str, Enum):
    COMPLETE = 'complete'
    TOOL_USE = 'tool_use'
    LENGTH = 'length'
    CONTENT_FILTER = 'content_filter'
    OTHER = 'other'


class ModelErrorCode(str, Enum):
    AUTHENTICATION = 'authentication'
    REQUEST = 'request'
    RATE_LIMIT = 'rate_limit'
    TIMEOUT = 'timeout'
    CONNECTION = 'connection'
    SERVER = 'server'
    PROVIDER = 'provider'


class UnsupportedContentError(ValueError):
    """Raised before provider I/O for a content variant outside version one."""


class ModelProtocolError(RuntimeError):
    """Raised when an adapter emits an incomplete provider-neutral stream."""


@dataclass(frozen=True, slots=True)
class TextContent:
    text: str
    type: Literal['text'] = field(default='text', init=False)
    schema_version: Literal[1] = field(default=1, init=False)

    def __post_init__(self) -> None:
        if not isinstance(self.text, str):
            raise TypeError('TextContent.text must be a string')


@dataclass(frozen=True, slots=True)
class ToolCallContent:
    id: str
    name: str
    arguments: str
    type: Literal['tool_call'] = field(default='tool_call', init=False)
    schema_version: Literal[1] = field(default=1, init=False)

    def __post_init__(self) -> None:
        if not self.id or not self.name:
            raise ValueError('a Tool Call requires non-empty id and name')
        if not isinstance(self.arguments, str):
            raise TypeError('ToolCallContent.arguments must be a JSON string')


ContentBlock: TypeAlias = TextContent | ToolCallContent


@dataclass(frozen=True, slots=True)
class AgentMessage:
    role: Role
    content: tuple[ContentBlock, ...]

    @classmethod
    def text(cls, role: Role, text: str) -> 'AgentMessage':
        return cls(role=role, content=(TextContent(text),))


@dataclass(frozen=True, slots=True)
class ModelMessage:
    role: Role
    content: tuple[ContentBlock, ...]


@dataclass(frozen=True, slots=True)
class ModelSpec:
    model_id: str
    context_window: int | None = None
    max_output_tokens: int = 4096
    supports_tools: bool = True

    def __post_init__(self) -> None:
        if not self.model_id.strip():
            raise ValueError('ModelSpec.model_id cannot be empty')
        if self.context_window is not None and self.context_window <= 0:
            raise ValueError('context_window must be positive when supplied')
        if self.max_output_tokens <= 0:
            raise ValueError('max_output_tokens must be positive')


@dataclass(frozen=True, slots=True)
class Usage:
    input_tokens: int
    output_tokens: int
    total_tokens: int
    estimated: bool = False


@dataclass(frozen=True, slots=True)
class ModelRequest:
    messages: tuple[ModelMessage, ...]
    model: ModelSpec


@dataclass(frozen=True, slots=True)
class TextDelta:
    text: str


@dataclass(frozen=True, slots=True)
class ToolCallDelta:
    index: int
    id: str = ''
    name: str = ''
    arguments_delta: str = ''


@dataclass(frozen=True, slots=True)
class UsageUpdate:
    usage: Usage


@dataclass(frozen=True, slots=True)
class ModelEnd:
    stop_reason: StopReason


ModelEvent: TypeAlias = TextDelta | ToolCallDelta | UsageUpdate | ModelEnd


@dataclass(frozen=True, slots=True)
class ModelResult:
    message: ModelMessage
    stop_reason: StopReason
    usage: Usage | None = None


@dataclass(frozen=True, slots=True)
class ModelError:
    code: ModelErrorCode
    message: str
    retryable: bool
    status_code: int | None = None


class ModelAdapterError(RuntimeError):
    def __init__(self, error: ModelError) -> None:
        self.error = error
        super().__init__(error.message)


class ModelAdapter(Protocol):
    def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]: ...


def _validate_content(block: object, role: Role) -> ContentBlock:
    if not isinstance(block, (TextContent, ToolCallContent)):
        raise UnsupportedContentError(
            'version one supports only TextContent and ToolCallContent'
        )
    if getattr(block, 'schema_version', None) != 1:
        raise UnsupportedContentError('unsupported Content Block schema version')
    if isinstance(block, ToolCallContent) and role is not Role.ASSISTANT:
        raise UnsupportedContentError('ToolCallContent is valid only for assistant messages')
    return block


def to_model_messages(messages: Sequence[AgentMessage]) -> tuple[ModelMessage, ...]:
    converted: list[ModelMessage] = []
    for message in messages:
        if not isinstance(message, AgentMessage):
            raise TypeError('model input must contain AgentMessage values')
        content = tuple(_validate_content(block, message.role) for block in message.content)
        converted.append(ModelMessage(role=message.role, content=content))
    return tuple(converted)


class ScriptedModelAdapter:
    """Replay a provider-neutral event script without credentials or I/O."""

    def __init__(self, events: Sequence[ModelEvent]) -> None:
        self._events = tuple(events)
        self._requests: list[ModelRequest] = []

    @property
    def received_requests(self) -> tuple[ModelRequest, ...]:
        return tuple(self._requests)

    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
        self._requests.append(request)
        for event in self._events:
            yield event


async def complete(
    adapter: ModelAdapter,
    messages: Sequence[AgentMessage],
    model: ModelSpec,
) -> ModelResult:
    request = ModelRequest(to_model_messages(messages), model)
    text_parts: list[str] = []
    tool_drafts: dict[int, dict[str, str]] = {}
    usage: Usage | None = None
    end: ModelEnd | None = None
    async for event in adapter.stream(request):
        if end is not None:
            raise ModelProtocolError('an adapter emitted data after ModelEnd')
        if isinstance(event, TextDelta):
            text_parts.append(event.text)
        elif isinstance(event, ToolCallDelta):
            if event.index < 0:
                raise ModelProtocolError('Tool Call indexes cannot be negative')
            draft = tool_drafts.setdefault(
                event.index, {'id': '', 'name': '', 'arguments': ''}
            )
            draft['id'] += event.id
            draft['name'] += event.name
            draft['arguments'] += event.arguments_delta
        elif isinstance(event, UsageUpdate):
            usage = event.usage
        elif isinstance(event, ModelEnd):
            end = event
        else:
            raise ModelProtocolError(f'unsupported model event: {type(event).__name__}')
    if end is None:
        raise ModelProtocolError('an adapter stream must end with ModelEnd')
    blocks: list[ContentBlock] = []
    if text_parts:
        blocks.append(TextContent(''.join(text_parts)))
    for index in sorted(tool_drafts):
        draft = tool_drafts[index]
        try:
            blocks.append(ToolCallContent(**draft))
        except (TypeError, ValueError) as error:
            raise ModelProtocolError(f'incomplete Tool Call at index {index}') from error
    return ModelResult(
        message=ModelMessage(Role.ASSISTANT, tuple(blocks)),
        stop_reason=end.stop_reason,
        usage=usage,
    )


@dataclass(frozen=True, slots=True)
class OpenAICompatibleConfig:
    base_url: str
    api_key: str
    headers: Mapping[str, str] = field(default_factory=dict)
    extra_body: Mapping[str, object] = field(default_factory=dict)
    timeout_seconds: float = 60.0

    def __post_init__(self) -> None:
        parsed = urlparse(self.base_url)
        if parsed.scheme not in {'http', 'https'} or not parsed.netloc:
            raise ValueError('base_url must be an explicit HTTP(S) URL')
        if not self.api_key:
            raise ValueError('api_key must be supplied explicitly')
        if self.timeout_seconds <= 0:
            raise ValueError('timeout_seconds must be positive')
        object.__setattr__(self, 'headers', MappingProxyType(dict(self.headers)))
        object.__setattr__(self, 'extra_body', MappingProxyType(dict(self.extra_body)))


def _provider_message(message: ModelMessage) -> dict[str, object]:
    text = ''.join(block.text for block in message.content if isinstance(block, TextContent))
    tool_calls = [block for block in message.content if isinstance(block, ToolCallContent)]
    encoded: dict[str, object] = {'role': message.role.value, 'content': text or None}
    if tool_calls:
        encoded['tool_calls'] = [
            {
                'id': block.id,
                'type': 'function',
                'function': {'name': block.name, 'arguments': block.arguments},
            }
            for block in tool_calls
        ]
    return encoded


def _stop_reason(value: str | None) -> StopReason:
    if value is None:
        return StopReason.OTHER
    return {
        'stop': StopReason.COMPLETE,
        'tool_calls': StopReason.TOOL_USE,
        'length': StopReason.LENGTH,
        'content_filter': StopReason.CONTENT_FILTER,
    }.get(value, StopReason.OTHER)


def _normalized_error(error: Exception) -> ModelError:
    status = getattr(error, 'status_code', None)
    if isinstance(error, openai.AuthenticationError):
        code, retryable = ModelErrorCode.AUTHENTICATION, False
    elif isinstance(error, openai.RateLimitError) or status == 429:
        code, retryable = ModelErrorCode.RATE_LIMIT, True
    elif isinstance(error, openai.APITimeoutError) or status == 408:
        code, retryable = ModelErrorCode.TIMEOUT, True
    elif isinstance(error, openai.APIConnectionError):
        code, retryable = ModelErrorCode.CONNECTION, True
    elif isinstance(status, int) and status >= 500:
        code, retryable = ModelErrorCode.SERVER, True
    elif isinstance(error, (openai.BadRequestError, openai.NotFoundError)):
        code, retryable = ModelErrorCode.REQUEST, False
    else:
        code, retryable = ModelErrorCode.PROVIDER, False
    status_text = f' with status {status}' if status is not None else ''
    return ModelError(
        code=code,
        message=f'OpenAI-compatible request failed{status_text}',
        retryable=retryable,
        status_code=status,
    )


class OpenAICompatibleAdapter:
    """Translate the streaming Chat Completions protocol at one seam."""

    def __init__(self, config: OpenAICompatibleConfig) -> None:
        self._config = config
        self._client = openai.AsyncOpenAI(
            api_key=config.api_key,
            base_url=config.base_url.rstrip('/') + '/',
            default_headers=dict(config.headers),
            timeout=config.timeout_seconds,
            max_retries=0,
        )

    async def stream(self, request: ModelRequest) -> AsyncIterator[ModelEvent]:
        finish_reason: str | None = None
        try:
            response = await self._client.chat.completions.create(
                model=request.model.model_id,
                messages=cast(list, [_provider_message(item) for item in request.messages]),
                max_tokens=request.model.max_output_tokens,
                stream=True,
                stream_options={'include_usage': True},
                extra_body=dict(self._config.extra_body) or None,
            )
            async for chunk in response:
                if chunk.usage is not None:
                    input_tokens = chunk.usage.prompt_tokens or 0
                    output_tokens = chunk.usage.completion_tokens or 0
                    total_tokens = chunk.usage.total_tokens or input_tokens + output_tokens
                    yield UsageUpdate(Usage(input_tokens, output_tokens, total_tokens))
                for choice in chunk.choices:
                    delta = choice.delta
                    if delta.content:
                        yield TextDelta(delta.content)
                    for tool_call in delta.tool_calls or ():
                        function = tool_call.function
                        yield ToolCallDelta(
                            index=tool_call.index,
                            id=tool_call.id or '',
                            name=(function.name if function else None) or '',
                            arguments_delta=(function.arguments if function else None) or '',
                        )
                    if choice.finish_reason is not None:
                        finish_reason = choice.finish_reason
        except openai.OpenAIError as error:
            raise ModelAdapterError(_normalized_error(error)) from None
        yield ModelEnd(_stop_reason(finish_reason))
'''

In [ ]:
exec(compile(MODEL_SOURCE, 'agent_harness/model.py', 'exec'))

scripted = ScriptedModelAdapter([
    TextDelta('Hello'),
    TextDelta(' from the offline model boundary.'),
    UsageUpdate(Usage(4, 7, 11)),
    ModelEnd(StopReason.COMPLETE),
])
minimal_result = await complete(
    scripted,
    [AgentMessage.text(Role.USER, 'Say hello')],
    ModelSpec('scripted/chapter-01'),
)
assert minimal_result.message.content == (
    TextContent('Hello from the offline model boundary.'),
)
minimal_result

## Staged Construction

The value objects form the public interface; `complete` is the deep module that owns validation and stream assembly. The adapter seam is justified by two implementations: deterministic scripted events for the course and explicit Chat Completions transport for production. `ModelSpec` carries capabilities separately from credentials and endpoint configuration.

In [ ]:
INIT_SOURCE = r'''"""Public interface for the Chapter 1 Agent Harness Checkpoint."""

from .model import (
    AgentMessage,
    ContentBlock,
    ModelAdapter,
    ModelAdapterError,
    ModelEnd,
    ModelError,
    ModelErrorCode,
    ModelEvent,
    ModelMessage,
    ModelProtocolError,
    ModelRequest,
    ModelResult,
    ModelSpec,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    ScriptedModelAdapter,
    StopReason,
    TextContent,
    TextDelta,
    ToolCallContent,
    ToolCallDelta,
    UnsupportedContentError,
    Usage,
    UsageUpdate,
    complete,
    to_model_messages,
)

__all__ = [name for name in globals() if not name.startswith('_')]
'''

In [ ]:
PY_TYPED_SOURCE = ''

## Observable Trace

Chapter 1 deliberately has no runtime event bus. Its smallest observable trace is the adapter's provider-neutral event script plus the exact typed request captured at the seam.

In [ ]:
trace = {
    'request_role': scripted.received_requests[0].messages[0].role.value,
    'event_types': ['TextDelta', 'TextDelta', 'UsageUpdate', 'ModelEnd'],
    'stop_reason': minimal_result.stop_reason.value,
    'usage': minimal_result.usage.total_tokens,
}
assert trace == {
    'request_role': 'user',
    'event_types': ['TextDelta', 'TextDelta', 'UsageUpdate', 'ModelEnd'],
    'stop_reason': 'complete',
    'usage': 11,
}
trace

## Failure Boundaries and Trade-offs

Unsupported content fails before provider I/O. The adapter performs one attempt and converts SDK failures to safe `ModelError` values; retry ownership arrives with `AgentRuntime` in Chapter 2. Tool arguments remain strings because schema parsing belongs to the Tool layer. The production adapter never reads environment or dotenv state.

In [ ]:
class ImageContent:
    schema_version = 1

try:
    to_model_messages([AgentMessage(Role.USER, (ImageContent(),))])
except UnsupportedContentError as error:
    assert 'TextContent and ToolCallContent' in str(error)
else:
    raise AssertionError('unsupported content was accepted')

The next Export Cells carry the offline conformance suite. A local HTTP server exercises the real SDK transport without external network access.

In [ ]:
MODEL_TEST_SOURCE = r'''from __future__ import annotations

import asyncio
from dataclasses import dataclass

import pytest

from agent_harness import (
    AgentMessage,
    ModelEnd,
    ModelMessage,
    ModelSpec,
    Role,
    ScriptedModelAdapter,
    StopReason,
    TextContent,
    TextDelta,
    ToolCallContent,
    ToolCallDelta,
    UnsupportedContentError,
    Usage,
    UsageUpdate,
    complete,
    to_model_messages,
)


def test_scripted_completion_crosses_the_typed_model_seam():
    adapter = ScriptedModelAdapter(
        [TextDelta('typed answer'), UsageUpdate(Usage(2, 3, 5)), ModelEnd(StopReason.COMPLETE)]
    )
    source = AgentMessage.text(Role.USER, 'hello')

    result = asyncio.run(complete(adapter, [source], ModelSpec('scripted/test')))

    assert result.message == ModelMessage(Role.ASSISTANT, (TextContent('typed answer'),))
    assert result.usage == Usage(2, 3, 5)
    request = adapter.received_requests[0]
    assert request.messages == (ModelMessage(Role.USER, (TextContent('hello'),)),)
    assert not isinstance(request.messages[0], dict)


def test_scripted_tool_argument_deltas_are_assembled_in_call_order():
    adapter = ScriptedModelAdapter(
        [
            ToolCallDelta(1, 'call-b', 'read', '{"path":'),
            ToolCallDelta(0, 'call-a', 'list', '{}'),
            ToolCallDelta(1, arguments_delta='"README.md"}'),
            ModelEnd(StopReason.TOOL_USE),
        ]
    )

    result = asyncio.run(complete(adapter, [], ModelSpec('scripted/tools')))

    assert result.message.content == (
        ToolCallContent('call-a', 'list', '{}'),
        ToolCallContent('call-b', 'read', '{"path":"README.md"}'),
    )


@dataclass(frozen=True)
class ImageContent:
    source: str = 'provider-image'
    schema_version: int = 1


def test_unsupported_content_fails_before_adapter_invocation():
    adapter = ScriptedModelAdapter([ModelEnd(StopReason.COMPLETE)])
    message = AgentMessage(Role.USER, (ImageContent(),))  # type: ignore[arg-type]

    with pytest.raises(UnsupportedContentError, match='only TextContent and ToolCallContent'):
        asyncio.run(complete(adapter, [message], ModelSpec('scripted/test')))

    assert adapter.received_requests == ()


def test_tool_calls_are_rejected_for_non_assistant_roles():
    message = AgentMessage(Role.USER, (ToolCallContent('call-1', 'read', '{}'),))

    with pytest.raises(UnsupportedContentError, match='assistant'):
        to_model_messages([message])
'''

In [ ]:
OPENAI_TEST_SOURCE = r'''from __future__ import annotations

import asyncio
from contextlib import contextmanager
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import json
from threading import Thread
from typing import Iterator

import pytest

from agent_harness import (
    AgentMessage,
    ModelAdapterError,
    ModelErrorCode,
    ModelSpec,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    StopReason,
    TextContent,
    ToolCallContent,
    Usage,
    complete,
)


def _chunk(delta, *, finish_reason=None):
    return {
        'id': 'chatcmpl-local',
        'object': 'chat.completion.chunk',
        'created': 1,
        'model': 'local-test',
        'choices': [
            {'index': 0, 'delta': delta, 'finish_reason': finish_reason}
        ],
    }


@contextmanager
def fake_openai_server() -> Iterator[tuple[str, list[dict[str, object]]]]:
    requests: list[dict[str, object]] = []

    class Handler(BaseHTTPRequestHandler):
        def do_POST(self):
            length = int(self.headers.get('content-length', '0'))
            body = json.loads(self.rfile.read(length))
            requests.append(
                {'path': self.path, 'authorization': self.headers.get('authorization'), 'body': body}
            )
            if body['model'] == 'reject-me':
                payload = json.dumps(
                    {'error': {'message': 'SECRET provider detail', 'type': 'invalid_request_error'}}
                ).encode()
                self.send_response(401)
                self.send_header('content-type', 'application/json')
                self.send_header('content-length', str(len(payload)))
                self.end_headers()
                self.wfile.write(payload)
                return

            events = [
                _chunk({'role': 'assistant', 'content': 'Hel'}),
                _chunk({'content': 'lo'}),
                _chunk(
                    {
                        'tool_calls': [
                            {
                                'index': 0,
                                'id': 'call-1',
                                'type': 'function',
                                'function': {'name': 'read', 'arguments': '{"path":'},
                            }
                        ]
                    }
                ),
                _chunk(
                    {'tool_calls': [{'index': 0, 'function': {'arguments': '"README.md"}'}}]}
                ),
                _chunk({}, finish_reason='tool_calls'),
                {
                    'id': 'chatcmpl-local',
                    'object': 'chat.completion.chunk',
                    'created': 1,
                    'model': 'local-test',
                    'choices': [],
                    'usage': {'prompt_tokens': 4, 'completion_tokens': 6, 'total_tokens': 10},
                },
            ]
            payload = ''.join(f'data: {json.dumps(event)}\n\n' for event in events)
            payload += 'data: [DONE]\n\n'
            encoded = payload.encode()
            self.send_response(200)
            self.send_header('content-type', 'text/event-stream')
            self.send_header('content-length', str(len(encoded)))
            self.end_headers()
            self.wfile.write(encoded)

        def log_message(self, format, *args):
            return

    server = ThreadingHTTPServer(('127.0.0.1', 0), Handler)
    thread = Thread(target=server.serve_forever, daemon=True)
    thread.start()
    try:
        host, port = server.server_address
        yield f'http://{host}:{port}/v1', requests
    finally:
        server.shutdown()
        server.server_close()
        thread.join(timeout=5)


def test_openai_compatible_stream_is_translated_at_the_adapter_seam(monkeypatch):
    monkeypatch.setenv('OPENAI_API_KEY', 'ambient-key-must-not-win')
    with fake_openai_server() as (base_url, requests):
        adapter = OpenAICompatibleAdapter(
            OpenAICompatibleConfig(base_url=base_url, api_key='explicit-key')
        )
        result = asyncio.run(
            complete(
                adapter,
                [AgentMessage.text(Role.USER, 'inspect')],
                ModelSpec('local-test', max_output_tokens=128),
            )
        )

    assert result.message.content == (
        TextContent('Hello'),
        ToolCallContent('call-1', 'read', '{"path":"README.md"}'),
    )
    assert result.stop_reason is StopReason.TOOL_USE
    assert result.usage == Usage(4, 6, 10)
    assert requests[0]['path'] == '/v1/chat/completions'
    assert requests[0]['authorization'] == 'Bearer explicit-key'
    assert requests[0]['body']['messages'] == [{'role': 'user', 'content': 'inspect'}]


def test_openai_compatible_failure_is_normalized_without_raw_detail():
    with fake_openai_server() as (base_url, _):
        adapter = OpenAICompatibleAdapter(
            OpenAICompatibleConfig(base_url=base_url, api_key='explicit-key')
        )
        with pytest.raises(ModelAdapterError) as captured:
            asyncio.run(
                complete(
                    adapter,
                    [AgentMessage.text(Role.USER, 'fail safely')],
                    ModelSpec('reject-me'),
                )
            )

    assert captured.value.error.code is ModelErrorCode.AUTHENTICATION
    assert captured.value.error.status_code == 401
    assert captured.value.error.retryable is False
    assert 'SECRET' not in str(captured.value)
'''

In [ ]:
SMOKE_TEST_SOURCE = r'''from __future__ import annotations

import asyncio
import os

import pytest

from agent_harness import (
    AgentMessage,
    ModelSpec,
    OpenAICompatibleAdapter,
    OpenAICompatibleConfig,
    Role,
    complete,
)


@pytest.mark.skipif(
    os.environ.get('AGENT_HARNESS_REAL_SMOKE') != '1',
    reason='set AGENT_HARNESS_REAL_SMOKE=1 to opt into the credential-gated smoke test',
)
def test_explicit_real_openai_compatible_endpoint():
    required = ('AGENT_HARNESS_BASE_URL', 'AGENT_HARNESS_API_KEY', 'AGENT_HARNESS_MODEL')
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        pytest.fail('missing explicit smoke configuration: ' + ', '.join(missing))
    adapter = OpenAICompatibleAdapter(
        OpenAICompatibleConfig(
            base_url=os.environ['AGENT_HARNESS_BASE_URL'],
            api_key=os.environ['AGENT_HARNESS_API_KEY'],
        )
    )
    result = asyncio.run(
        complete(
            adapter,
            [AgentMessage.text(Role.USER, 'Reply with the word ready.')],
            ModelSpec(os.environ['AGENT_HARNESS_MODEL'], max_output_tokens=32),
        )
    )
    assert result.message.content
'''

## Checkpoint Export and Verification

The remaining Export Cells define the installable distribution. The final cell stages every tagged file, compiles it, installs without dependency resolution or network access, imports it, runs all exported tests, writes a deterministic manifest, and only then publishes Chapter 1.

In [ ]:
PYPROJECT_SOURCE = r'''[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "agent-harness"
version = "0.1.0"
description = "Chapter 1 provider-neutral model boundary"
readme = "README.md"
requires-python = ">=3.11"
dependencies = ["openai>=1.40,<3"]

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
agent_harness = ["py.typed"]

[tool.pytest.ini_options]
testpaths = ["tests"]
'''

In [ ]:
README_SOURCE = r'''# Agent Harness — Chapter 1 Checkpoint

This immutable Checkpoint contains the typed provider-neutral model seam built by Chapter 1. It supports Python 3.11 or newer and one production transport: explicitly configured OpenAI-compatible streaming Chat Completions.

Ordinary tests use `ScriptedModelAdapter` or the local fake endpoint. The real-endpoint smoke test runs only when `AGENT_HARNESS_REAL_SMOKE=1` and all three explicit endpoint variables are supplied.
'''

In [ ]:
from pathlib import Path

from course.tools.checkpoint import export_checkpoint

repository = Path.cwd().resolve()
chapter = repository / 'course' / 'notebooks' / '01_model_boundary.ipynb'
checkpoint = repository / 'course' / 'checkpoints' / 'ch01'
export_result = export_checkpoint(chapter, checkpoint)
assert export_result.gates == ('compile', 'install', 'import', 'tests')
assert export_result.digest
export_result

## Public API Summary

Callers construct `AgentMessage` and `ModelSpec`, choose either `ScriptedModelAdapter` or `OpenAICompatibleAdapter(OpenAICompatibleConfig(...))`, and await `complete`. The result is a provider-neutral `ModelResult` containing versioned `TextContent` and `ToolCallContent`, normalized `StopReason`, and optional reported `Usage`. Configuration is explicit and no singleton or terminal I/O is involved.